## 1. Imports & Environment Setup 

In [2]:
# Imports
from pathlib import Path
import json
import pandas as pd
import sys, os
from dotenv import load_dotenv
from mozzarellm.pipeline.bundle_builder import (
    load_table,
    load_screen_context_json,
    build_evidence_bundles,
    get_or_append_stable_accession,
)

print(sys.path)

['C:\\Users\\alexa\\AppData\\Local\\Python\\pythoncore-3.12-64\\python312.zip', 'C:\\Users\\alexa\\AppData\\Local\\Python\\pythoncore-3.12-64\\DLLs', 'C:\\Users\\alexa\\AppData\\Local\\Python\\pythoncore-3.12-64\\Lib', 'C:\\Users\\alexa\\AppData\\Local\\Python\\pythoncore-3.12-64', 'c:\\Users\\alexa\\OneDrive - Massachusetts Institute of Technology\\Cheesman Lab\\WORKSPACE\\mozzarellm\\.mozzvenv', '', 'c:\\Users\\alexa\\OneDrive - Massachusetts Institute of Technology\\Cheesman Lab\\WORKSPACE\\mozzarellm\\.mozzvenv\\Lib\\site-packages', '__editable__.mozzarellm-0.2.0.finder.__path_hook__']


In [3]:
def _find_repo_root(start_dir: Path) -> Path:
    start_dir = start_dir.resolve()
    for p in [start_dir] + list(start_dir.parents):
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError(f"Could not find pyproject.toml starting from {start_dir}")


# Use repo root for sys.path and for locating .env, regardless of notebook working directory (less brittle).
repo_root = _find_repo_root(Path(os.getcwd()))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Load secrets/config from .env at the repo root (if present)
print("Does .env exist in repo root?", (repo_root / ".env").exists())
load_dotenv(dotenv_path=repo_root / ".env")
print("Repo root:", repo_root)
print("Current working directory:", os.getcwd())

Does .env exist in repo root? True
Repo root: C:\Users\alexa\OneDrive - Massachusetts Institute of Technology\Cheesman Lab\WORKSPACE\mozzarellm
Current working directory: c:\Users\alexa\OneDrive - Massachusetts Institute of Technology\Cheesman Lab\WORKSPACE\mozzarellm\interface


## 2. Screen Context Configuration

Complete the `screen_context.json` file with the relevant information for your screen. Adjust the template as needed. Top level must be a dictionary. 

In [4]:
# set screen name
SCREEN_NAME = "test"
SCREEN_CONTEXT_PATH = Path("screen_context.json")
screen_ctx = load_screen_context_json(SCREEN_CONTEXT_PATH)
# should throw error if file doesn't exist or if you forgot to remove the TODO!

print("Loaded screen_context.json:", isinstance(screen_ctx, dict))
keys = list(screen_ctx.keys())
print(f"Top-level keys ({len(keys)}): {keys}")

Loaded screen_context.json: True
Top-level keys (10): ['assay_type', 'target_phenotype', 'organism', 'cell_line_or_system', 'perturbation', 'readout', 'clustering', 'controls', 'provenance', 'notes']


## 3. Load Cluster Table
Set parameters and check that the cluster table loads.


In [5]:
# 3.1 Point this to your cluster/gene table (CSV/TSV/TXT/XLSX)
default_cluster_table = (
    repo_root / "examples" / "ops" / "funk_2022.csv"
)  # NOTE: using funk_2022.csv for development
CLUSTER_TABLE_PATH = Path(default_cluster_table)

#### 3.2 Optional: #####
sep = None
# Useful for .txt files. If the table loads as one big column (or in an otherwise wonky fashion), the delimiter is likely mismatched.
# By default pandas ASSUMES the delimiter (value separator) based on file extension
# (typically .csv → ',', .tsv → '\t'). Common overrides are ',', '|', '\t', or ';'.

sheet_name = None
# For Excel only (e.g. 0 or 'Sheet1'); ignored for CSV/TSV
########################

cluster_df = load_table(CLUSTER_TABLE_PATH, sep=sep, sheet_name=sheet_name)
print("Loaded:", str(CLUSTER_TABLE_PATH))
print("Shape:", cluster_df.shape)
print("Columns:\n", list(cluster_df.columns))
display(cluster_df.head())

Loaded: C:\Users\alexa\OneDrive - Massachusetts Institute of Technology\Cheesman Lab\WORKSPACE\mozzarellm\examples\ops\funk_2022.csv
Shape: (164, 5)
Columns:
 ['gene_symbol', 'cluster', 'up_features', 'down_features', 'phenotypic_strength']


,gene_symbol,cluster,up_features,down_features,phenotypic_strength
0,AATF,21,"interphase_cell_correlation_dapi_tubulin,inter...","interphase_nucleus_area,interphase_cell_area,i...",669/5299
1,ABT1,21,"interphase_nucleus_gh2ax_mean,interphase_nucle...","interphase_nucleus_area,interphase_cell_area,i...",560/5299
2,BMS1,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",446/5299
3,BYSL,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",524/5299
4,C1orf131,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",343/5299


In [11]:
# set cluster id column name or keep default
CLUSTER_ID_COLUMN = "cluster"
# set gene column name or keep default
GENE_COLUMN = "gene_symbol"
# if you have a stable accession column in your cluster table, set it here
STABLE_ACCESSION_COLUMN = None  # or the name of your stable accession column
ORGANISM_ID = 9606  # human

# if you do not have stable accession IDs in your cluster table, we will assign them.
# You can also provide a table that contains a gene to stable accession mapping, and we will append them to your cluster table.
# A csv copy of the gene to stable accession mapping is provided in the interface/output/
if STABLE_ACCESSION_COLUMN is None:
    acc_cluster_df = get_or_append_stable_accession(
        cluster_df=cluster_df,
        accession_table=None,  # or Path("path/to/accession_table.csv")
        accession_col=None,  # or the name of your stable accession column in the seperate table
        accession_table_sep=None,  # or the separator for the seperate table
        accession_table_sheetname=None,  # or the sheet name for the seperate table if it's an excel file
        organism_id=ORGANISM_ID,
        warn_on_fallback=False,  # ignore warnings for unreviewed entries
    )
# NOTE: running this cell will overwrite the last .csv output
acc_cluster_df

,gene_symbol,cluster,up_features,down_features,phenotypic_strength,accession
0,AATF,21,"interphase_cell_correlation_dapi_tubulin,inter...","interphase_nucleus_area,interphase_cell_area,i...",669/5299,Q9NY61
1,ABT1,21,"interphase_nucleus_gh2ax_mean,interphase_nucle...","interphase_nucleus_area,interphase_cell_area,i...",560/5299,Q9ULW3
2,BMS1,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",446/5299,Q14692
3,BYSL,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",524/5299,Q13895
4,C1orf131,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",343/5299,Q8NDD1
...,...,...,...,...,...,...
159,nontargeting_5_2,99,NaN,NaN,5203/5299,NON_TARGETING_CONTROL
160,nontargeting_61_0,99,NaN,NaN,5038/5299,NON_TARGETING_CONTROL
161,nontargeting_6_1,99,NaN,interphase_cell_phalloidin_mean,5237/5299,NON_TARGETING_CONTROL
162,nontargeting_8_2,99,NaN,NaN,5271/5299,NON_TARGETING_CONTROL


## 4. Create Per-Cluster Evidence Bundles

In [7]:
BUNDLE_OUTPUT_PATH_BASE = f"{repo_root}/interface/output/intermediates/{SCREEN_NAME}__bundles/"
build_evidence_bundles(
    screen_name=SCREEN_NAME,
    cluster_id_column=CLUSTER_ID_COLUMN,
    gene_column=GENE_COLUMN,
    top_k=5,
)

AttributeError: 'NoneType' object has no attribute 'columns'